# ਪਾਠ 09 - ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ


## ਸੈਟਅਪ

ਇਹ ਨੋਟਬੁੱਕ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਮੈਟਾਕੋਗਨੀਸ਼ਨ ਡਿਜ਼ਾਈਨ ਪੈਟਰਨ ਨੂੰ ਦਰਸਾਉਂਦਾ ਹੈ।

**ਜਰੂਰੀ ਸ਼ਰਤਾਂ:**
- Azure OpenAI ਡਿਪਲੋਇਮੈਂਟ environment variables ਰਾਹੀਂ ਕਨਫਿਗਰ ਕੀਤੀ ਹੋਈ ਹੋਵੇ
- Azure CLI ਪ੍ਰਮਾਣਿਤ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਕੀ ਹੈ?

ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਦਾ ਮਤਲਬ ਹੈ **ਸੋਚ ਬਾਰੇ ਸੋਚਣਾ**। ਕ੍ਰਿਤ੍ਰਿਮ ਬੁੱਧੀ ਏਜੰਟਾਂ ਦੇ ਸੰਦਰਭ ਵਿੱਚ, ਇਸਦਾ ਮਤਲਬ ਹੈ ਐਸੇ ਏਜੰਟ ਬਣਾਉਣਾ ਜੋ ਸਕਦੇ ਹਨ:

- **ਆਤਮ-ਚਿੰਤਨ** ਆਪਣੇ ਨਤੀਜਿਆਂ ਅਤੇ ਤਰਕ ਪ੍ਰਕਿਰਿਆ ਤੇ
- **ਗਲਤੀਆਂ ਪਹਿਚਾਣਣਾ** ਅਤੇ ਚੁੱਪਚਾਪ ਨਾਕਾਮ ਹੋਣ ਦੀ ਥਾਂ ਸੁਸ਼ੀਲਤਾਪੂਰਵਕ ਮੁੜ-ਬਹਾਲ ਹੋਣਾ
- **ਮੁਲਿਆੰਕਣ** ਕਿ ਕੀ ਉਹਨਾਂ ਦੇ ਜਵਾਬ ਪੂਰੇ ਅਤੇ ਮਦਦਗਾਰ ਹਨ
- **ਅਨੁਕੂਲ ਹੋਣਾ** ਉਹਨਾਂ ਦੀ ਰਣਨੀਤੀ ਨੂੰ ਜਦੋਂ ਸ਼ੁਰੂਆਤੀ ਤਰੀਕਾ ਕੰਮ ਨਾ ਕਰੇ (ਉਦਾਹਰਨ ਲਈ, ਬੈਕਅੱਪ ਸਿਸਟਮ ਵੱਲ ਮੁੜਣਾ)

ਇੱਕ ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਵਾਲਾ ਏਜੰਟ ਸਿਰਫ ਸਵਾਲਾਂ ਦੇ ਜਵਾਬ ਨਹੀਂ ਦਿੰਦਾ — ਇਹ ਆਪਣੀ ਕਾਰਗੁਜ਼ਾਰੀ ਦੀ ਨਿਗਰਾਨੀ ਕਰਦਾ ਹੈ ਅਤੇ ਤੁਰੰਤ ਢਾਲ ਲੈਂਦਾ ਹੈ।


## ਮੁੱਖ ਅਤੇ ਬੈਕਅੱਪ ਟੂਲ

ਇੱਕ ਆਮ ਮੈਟਾਕੌਗਨੀਸ਼ਨ ਪੈਟਰਨ **ਵਿਕਲਪਿਕ ਰਣਨੀਤੀ** ਹੈ। ਏਜੰਟ ਪਹਿਲਾਂ ਇੱਕ ਮੁੱਖ ਟੂਲ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰਦਾ ਹੈ; ਜੇ ਇਹ ਫੇਲ ਹੋ ਜਾਏ (ਉਦਾਹਰਨ ਲਈ, 404 ਐਰਰ), ਤਾਂ ਏਜੰਟ ਨਾਕਾਮੀ ਨੂੰ ਪਛਾਣਦਾ ਹੈ ਅਤੇ ਸਪਸ਼ਟ ਤੌਰ 'ਤੇ ਬੈਕਅੱਪ ਟੂਲ ਵੱਲ ਬਦਲ ਜਾਂਦਾ ਹੈ।

ਇਹ ਅਸਲੀ ਦੁਨੀਆ ਦੀਆਂ ਪ੍ਰਣਾਲੀਆਂ ਦੀ ਨਕਲ ਕਰਦਾ ਹੈ ਜਿੱਥੇ ਮੁੱਖ ਸੇਵਾਵਾਂ ਉਪਲਬਧ ਨਹੀਂ ਹੋ ਸਕਦੀਆਂ ਅਤੇ ਏਜੰਟਾਂ ਨੂੰ ਵਿਕਲਪਿਕ ਰਾਹ ਚੁਣਨ ਤੋਂ ਪਹਿਲਾਂ ਸਮੱਸਿਆ ਦੀ ਖੁਦ-ਪਛਾਣ ਕਰਨੀ ਪੈਂਦੀ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਦੋ ਫਲਾਈਟ ਲੁੱਕਅਪ ਟੂਲ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ:
- **ਮੁੱਖ** — ਪੈਰਿਸ, ਟੋਕੀਓ, ਅਤੇ ਬਾਰਸਿਲੋਨਾ ਨੂੰ ਕਵਰ ਕਰਦਾ ਹੈ
- **ਬੈਕਅੱਪ** — ਬਰਲਿਨ, ਸਿਡਨੀ, ਅਤੇ ਨਿਊ ਯਾਰਕ ਸਿਟੀ ਨੂੰ ਕਵਰ ਕਰਦਾ ਹੈ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ਗਲਤੀ ਪੁਨਰਪ੍ਰਾਪਤੀ ਵਾਲਾ ਸਵ-ਅਵਲੋਕਨ ਏਜੰਟ

ਹੇਠਾਂ ਦਿੱਤੇ ਏਜੰਟ ਨੂੰ ਹੁਕਮ ਦਿੱਤਾ ਗਿਆ ਹੈ ਕਿ ਪਹਿਲਾਂ ਮੁੱਖ ਉਡਾਣ ਪ੍ਰਣਾਲੀ ਦੀ ਕੋਸ਼ਿਸ਼ ਕਰੇ, ਨਾਕਾਮੀਆਂ ਨੂੰ ਪਛਾਣੇ, ਅਤੇ ਪਾਰਦਰਸ਼ੀ ਤਰੀਕੇ ਨਾਲ ਬੈਕਅਪ ਪ੍ਰਣਾਲੀ 'ਤੇ ਵਾਪਸ ਆ ਜਾਵੇ। ਹਰ ਜਵਾਬ ਤੋਂ ਬਾਅਦ ਇਹ ਸੰਖੇਪ ਵਿੱਚ ਖੁਦ-ਮੁਲਾਂਕਣ ਕਰਦਾ ਹੈ ਕਿ ਕੀ ਇਸ ਨੇ ਯੂਜ਼ਰ ਦੇ ਸਵਾਲ ਦਾ ਪੂਰਾ ਜਵਾਬ ਦਿੱਤਾ।


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ਆਤਮ-ਮੂਲਾਂਕਣ ਪੈਟਰਨ

ਮੇਟਾਕੋਗਨੀਸ਼ਨ ਦਾ ਇੱਕ ਹੋਰ ਪਹਲੂ **ਆਤਮ-ਮੂਲਾਂਕਣ** ਹੈ: ਇੱਕ ਵੱਖਰਾ ਏਜੰਟ (ਜਾਂ ਦੂਜੇ ਪਾਸ ਵਿੱਚ ਉਹੀ ਏਜੰਟ) ਇੱਕ ਜਵਾਬ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਪਣ ਅਤੇ ਮਦਦਗਾਰਤਾ ਲਈ ਸਮੀਖਿਆ ਕਰਦਾ ਹੈ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ `ResponseEvaluator` ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜੋ ਟ੍ਰੈਵਲ-ਏਜੰਟ ਦੇ ਜਵਾਬਾਂ ਨੂੰ ਤਿੰਨ ਮਾਪਦੰਡਾਂ 'ਤੇ ਸਕੋਰ ਕਰਦਾ ਹੈ।


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ **ਮੈਟਾਕਾਗਨਿਟਿਵ ਏਜੰਟ** ਕਿਵੇਂ ਬਣਾਉਣੇ ਸਿੱਖੇ:

- **ਆਤਮ-ਚਿੰਤਨ**: ਏਜੰਟ ਆਪਣੇ ਸੋਚਣ-ਵਿਚਾਰ ਦੀ ਨਿਗਰਾਨੀ ਕਰਦੇ ਹਨ ਅਤੇ ਪਾਰਦਰਸ਼ੀ ਢੰਗ ਨਾਲ ਦੱਸਦੇ ਹਨ ਕਿ ਕੀ ਹੋਇਆ।
- **ਫੈਲਬੈਕ ਨਾਲ ਗਲਤੀ ਰਿਕਵਰੀ**: ਇੱਕ ਮੁੱਖ + ਬੈਕਅੱਪ ਟੂਲ ਪੈਟਰਨ ਜਿਸ ਵਿੱਚ ਏਜੰਟ ਅਸਫਲਤਾਵਾਂ ਦੀ ਪਹਚਾਨ ਕਰਦਾ ਹੈ (ਉਦਾਹਰਨ ਲਈ, 404 ਗਲਤੀਆਂ) ਅਤੇ ਆਪਣੇ ਆਪ ਕਿਸੇ ਵਿਕਲਪੀ ਸਰੋਤ ਨੂੰ ਅਜ਼ਮਾਉਂਦਾ ਹੈ।
- **ਆਤਮ-ਮੁਲਾਂਕਣ**: ਇੱਕ ਵੱਖਰਾ ਮੁਲਾਂਕਣ ਕਰਨ ਵਾਲਾ ਏਜੰਟ ਜੋ ਜਵਾਬਾਂ ਨੂੰ ਪੂਰਨਤਾ, ਸਹੀਤਾ, ਅਤੇ ਮਦਦਗਾਰਤਾ ਲਈ ਸਕੋਰ ਕਰਦਾ ਹੈ।

ਇਹ ਪੈਟਰਨ ਏਜੰਟਾਂ ਨੂੰ ਹੋਰ ਮਜ਼ਬੂਤ, ਪਾਰਦਰਸ਼ੀ, ਅਤੇ ਭਰੋਸੇਯੋਗ ਬਣਾਉਂਦੇ ਹਨ — ਉਤਪਾਦਨ ਤੈਨਾਤੀਆਂ ਲਈ ਇਹ ਅਹਮ ਗੁਣ ਹਨ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਵੀਕਰਨ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਹਾਲਾਂਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਯਤਨ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਉਸ ਦੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਧਿਕਾਰਿਤ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਣ ਜਾਣਕਾਰੀ ਲਈ ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਕਰਨ ਨਾਲ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
